In [3]:
import bibtexparser
import numpy as np
import pandas as pd

from bibtexparser.bparser import BibTexParser
from pathlib import Path
import gzip
import re
import unicodedata

In [5]:
# OLD PROMPTS
# LLM
JUDGE_AXES_1 = [
    ("language_specific",
     "names a specific language, language family, or community as the focus "
     "(e.g. 'Thai', 'Yoruba'). Generic 'multilingual' without naming languages "
     "does NOT count."),
    ("linguistic_phenomenon",
     "centers a specific linguistic phenomenon or theory (e.g. coreference, "
     "morphology, discourse, syntax). Treating language as generic input/output "
     "to a model does NOT count."),
    ("benchmark_sota_framing",
     "frames the contribution primarily as benchmark performance, beating "
     "baselines, or achieving SOTA."),
    ("llm_as_object",
     "treats a large language model as the object of study (probing, prompting, "
     "or evaluating its capabilities). If an LLM is just a tool to study "
     "language, answer 0."),
    ("reasoning_cognition_claims",
     "makes claims about reasoning, understanding, intelligence, emergent "
     "capability, or human-level / general cognition."),
    ("societal_impact",
     "explicitly considers societal impact, affected communities, ethics, "
     "bias, fairness, or who is helped or harmed."),
]

JUDGE_SYSTEM_1 = """
Please act as an impartial judge and evaluate whether the provided title and abstract align with the given criterion.

Focus solely on whether the abstract and title meet the criterion. Do not make subjective judgments about the quality or significance of the research. Use only the information present in the title and abstract. Do not rely on outside knowledge, unstated assumptions, or weak implications that are not clearly supported by the text.

Evaluation guidelines:
- A score of 0 means the criterion is absent. The title and abstract do not explicitly mention it, and it is not necessary to describe the paper's main contribution.
- A score of 1 means the criterion is present but secondary. The criterion is mentioned, used, analyzed, or evaluated, but it is not the primary focus or central contribution of the paper.
- A score of 2 means the criterion is central. The criterion is a primary focus, main contribution, or organizing theme of the paper, and the title or abstract explicitly emphasizes it.

When evaluating:
1. Begin by providing a short explanation of whether the abstract and title clearly express the given criterion.
2. Point out explicit references to the criterion in the text, if any.
3. End with the rating in exactly this format: Rating: [[score]]

The score must be 0, 1, or 2. Do not output anything after the rating."""

PROMPT_TEMPLATE_1 = """Title: {title}\n\nAbstract: {abstract}

Criterion: This study {axis_desc}"""

In [6]:
# NEW PROMPTS
JUDGE_AXES = [
(
    "language_specific",
    "Language Named: Does the paper name the languages it studies?",
    "Score 0: A score of 0 means the criterion is absent. Does not name any language (note that the words multilingual and translation do not count as naming a language)\nScore 1: A score of 1 means the criterion is present but secondary. Names a language but core contributions of paper also depend on work done in other languages that are not mentioned\nScore 2: A score of 2 means the criterion is central. Names all languages in the paper relevant to its core contributions (note that specifying a paper deals with English counts)"
),
(
    "linguistic_phenomenon",
    "Linguistic Phenomenon: Does the paper study some linguistic phenomenon?",
    "Score 0: A score of 0 means the criterion is absent.  No mention of linguistics, or mentions semantics but has no grounding in linguistic theory\nScore 1: A score of 1 means the criterion is present but secondary. Uses linguistic knowledge to achieve a separate goal rather than to advance linguistics\nScore 2: A score of 2 means the criterion is central. Studies a linguistic phenomenon"
),

(
    "benchmark_sota_framing",
    "SOTA: Does the paper frame its contribution as improving performance on some task compared to baselines?",
    "Score 0: A score of 0 means the criterion is absent. No mention of evaluation, or creates an evaluation for the purpose of diagnosing a problem in a model\nScore 1: A score of 1 means the criterion is present but secondary. Discuses performance of method/system compared to baselines but the purpose of the paper is not to create the best possible method/system, or main contribution is the creation of a benchmark intended to be used by others to demonstrate their methods are SOTA\nScore 2: A score of 2 means the criterion is central. Claims method/system now has the best (or near best) performance and is SOTA, and being SOTA is a main contribution of the paper"
),

(
    "llm_as_object",
    "LLM-as-object: Does the paper frame its contribution as learning something about an LLM/pretrained model?",
    "Score 0: A score of 0 means the criterion is absent. No inclusion of LLM/pretrained, or LLM/pretrained is only included as a baseline, or LLM/pretrained is used as a method without the intention to learn something about it\nScore 1: A score of 1 means the criterion is present but secondary. Learns something about LLM/pretrained but this isn't the main contribution\nScore 2: A score of 2 means the criterion is central. The main contribution is to learn something about an LLM/pretrained (e.g. evaluating, interpreting, capability) or a paper describes building a new LLM/pretrained model"
),

(
    "reasoning_cognition_claims",
    "Reasoning cognition claims: Does the paper make claims about cognition, reasoning, understanding or emergence in a system?",
    "Score 0: A score of 0 means the criterion is absent. Not about cognition/reasoning/understanding/emergence or makes such claims but they are not about a system\nScore 1: A score of 1 means the criterion is present but secondary. States that the task studied requires cognition/reasoning/understanding/emergence but no direct claim is made about a system\nScore 2: A score of 2 means the criterion is central. Makes claims about cognition/reasoning/understanding/emergence in a system"
),

(
    "societal_impact",
    "Societal Impact: Does the paper frame its contribution as societal impact?",
    "Score 0: A score of 0 means the criterion is absent. No discussion of the social impact\nScore 1: A score of 1 means the criterion is present but secondary. Societal impact is mentioned but is not the main contribution\nScore 2: A score of 2 means the criterion is central. Purpose or main contribution deals with social impact"
),
]

JUDGE_SYSTEM = """Your input fields are:
1. `title` (str): Paper title
2. `abstract` (str): Paper abstract
Your output fields are:
1. `explanation` (str): A short explanation of whether the abstract and title clearly express the given criterion
2. `rating` (int): One integer only: 0, 1, or 2

Please act as an impartial judge and evaluate whether the provided title and abstract align with the criterion: {axis_desc}

Focus solely on whether the abstract and title meet the criterion. Do not make subjective judgments about the quality or significance of the research. Use only the information present in the title and abstract. Do not rely on outside knowledge, unstated assumptions, or weak implications that are not clearly supported by the text.

Evaluation guidelines:
{evaluation_guidelines}

When evaluating:
1. Begin by providing a short explanation of whether the abstract and title clearly express the given criterion.
2. Point out explicit references to the criterion in the text, if any.
3. End with the rating in exactly this format: Rating: [[score]]

The score must be 0, 1, or 2. Do not output anything after the rating."""

PROMPT_TEMPLATE = """Title: {title}

Abstract: {abstract}"""

In [7]:
def clean_latex(s: str) -> str:
    """Strip LaTeX accents and braces; normalize unicode and whitespace."""
    if not s:
        return ""
    s = re.sub(r"\\[`'^\"~=.]\{?([A-Za-z])\}?", r"\1", s)  # \'e -> e
    s = re.sub(r"\\[A-Za-z]+\{([^}]*)\}", r"\1", s)         # \emph{x} -> x
    s = s.replace("{", "").replace("}", "").replace("\\", "")
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s).strip()

In [8]:
def load_csv(data_file):
    data = pd.read_csv(data_file)
    papers = []
    n_no_abstract = 0
    for _, e in data.iterrows():
        abstract = clean_latex(e.get("Abstract", ""))
        if not abstract:
            n_no_abstract += 1
            continue
        
        try:
            year = int(e.get("Year"))
        except ValueError:
            continue
        
        papers.append({
            "key": e.get("ID", ""),
            "year": year,
            "title": clean_latex(e.get("Title", "")),
            "abstract": abstract,
            "venue": e.get("Venue", ""),
            "award": e.get("Award", "")
        })
    return papers

In [9]:
def get_prompts_prev(papers):
    prompts = {}
    for axis_key, _ in JUDGE_AXES_1:
        prompts[axis_key] = []
    
    for paper in papers:
        paper_title = paper['title']
        paper_abstract = paper['abstract']
        for axis_key, axis_desc in JUDGE_AXES_1:
            system_prompt = JUDGE_SYSTEM_1

            user_prompt = PROMPT_TEMPLATE_1.format(
                title=paper_title,
                abstract=paper_abstract,
                axis_desc=axis_desc
            )
            prompts[axis_key].append({"system_prompt": system_prompt, "user_prompt": user_prompt})
    return prompts

def get_prompts_new(papers):
    prompts = {}
    for axis_key, _, _ in JUDGE_AXES:
        prompts[axis_key] = []
    
    for paper in papers:
        paper_title = paper['title']
        paper_abstract = paper['abstract']
        for axis_key, axis_desc, evaluation_guidelines in JUDGE_AXES:
            system_prompt = JUDGE_SYSTEM.format(
                axis_desc=axis_desc,
                evaluation_guidelines=evaluation_guidelines,
            )

            user_prompt = PROMPT_TEMPLATE.format(
                title=paper_title,
                abstract=paper_abstract
            )
            prompts[axis_key].append({"system_prompt": system_prompt, "user_prompt": user_prompt})
    return prompts

In [10]:
papers = load_csv("../splits/best_papers_acl_full.csv")
prompts_prev = get_prompts_prev(papers)
prompts_new = get_prompts_new(papers)

In [11]:
from openai import OpenAI

client = OpenAI(api_key=API_KEY)

In [23]:
def get_response(prompt):
    response = client.responses.create(
        model="gpt-5.4-mini",
        instructions=prompt['system_prompt'],
        input=prompt['user_prompt'],
    )
    return response

def get_token_count(prompt):
    count = client.responses.input_tokens.count(
        model="gpt-5.4-mini",
        instructions=prompt['system_prompt'],
        input=prompt['user_prompt'],
    )
    return count

In [14]:
prompts_prev['language_specific'][0]

{'system_prompt': "\nPlease act as an impartial judge and evaluate whether the provided title and abstract align with the given criterion.\n\nFocus solely on whether the abstract and title meet the criterion. Do not make subjective judgments about the quality or significance of the research. Use only the information present in the title and abstract. Do not rely on outside knowledge, unstated assumptions, or weak implications that are not clearly supported by the text.\n\nEvaluation guidelines:\n- A score of 0 means the criterion is absent. The title and abstract do not explicitly mention it, and it is not necessary to describe the paper's main contribution.\n- A score of 1 means the criterion is present but secondary. The criterion is mentioned, used, analyzed, or evaluated, but it is not the primary focus or central contribution of the paper.\n- A score of 2 means the criterion is central. The criterion is a primary focus, main contribution, or organizing theme of the paper, and the 

In [15]:
response = get_response(prompts_prev['language_specific'][0])

In [24]:
count = get_token_count(prompts_prev['language_specific'][0])

In [25]:
count

InputTokenCountResponse(input_tokens=547, object='response.input_tokens')

In [17]:
print(response)

Response(id='resp_0cc02828dd1d8802006a132e6b43c08192956952c99387c47d', created_at=1779641963.0, error=None, incomplete_details=None, instructions="\nPlease act as an impartial judge and evaluate whether the provided title and abstract align with the given criterion.\n\nFocus solely on whether the abstract and title meet the criterion. Do not make subjective judgments about the quality or significance of the research. Use only the information present in the title and abstract. Do not rely on outside knowledge, unstated assumptions, or weak implications that are not clearly supported by the text.\n\nEvaluation guidelines:\n- A score of 0 means the criterion is absent. The title and abstract do not explicitly mention it, and it is not necessary to describe the paper's main contribution.\n- A score of 1 means the criterion is present but secondary. The criterion is mentioned, used, analyzed, or evaluated, but it is not the primary focus or central contribution of the paper.\n- A score of 2

In [19]:
print(response.output_text)

The title and abstract do not clearly name a specific language, language family, or community as the paper’s focus. The title only refers broadly to “Neural Machine Translation,” and the abstract mentions examples like English-German translation and TED multilingual translation, but these are task/dataset examples rather than a named language, language family, or community as the central focus. Since no single specific language or community is explicitly presented as the study’s focus, the criterion is absent.

Rating: [[0]]


In [20]:
print(response.usage)

ResponseUsage(input_tokens=547, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=103, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=650)


In [21]:
usage = response.usage

input_tokens = usage.input_tokens
output_tokens = usage.output_tokens

cost = (
    input_tokens * 0.75 / 1_000_000
    + output_tokens * 4.50 / 1_000_000
)

print(f"Estimated cost: ${cost:.6f}")

Estimated cost: $0.000874
